# Cuiman Job Result Openers

This section demonstrates how use `client.open_job_results()` and to enhance
the method's capabilities by adding a custom `JobResultOpener` to the client.

We use a test server here:

```bash
wraptile run -- wraptile.services.local.testing:service
```

In [ ]:
from cuiman import Client

In [2]:
client = Client()
# client

In [3]:
# client.get_processes()

We'd like to open the output of a process that outputs something to the filesystem, 
we use `"simulate_scene"` here. It has a `output_path` parameter for that purpose.

In [4]:
client.get_process(process_id="simulate_scene")

ProcessDescription(title='Generate scene for testing', description='Simulate a set scene images slices for testing. Creates an xarray dataset with `periodicity` time slices and writes it as Zarr into a temporary location. Requires installed `dask`, `xarray`, and `zarr` packages.', keywords=None, metadata=None, additionalParameters=None, id='simulate_scene', version='0.0.0', jobControlOptions=None, outputTransmission=None, links=None, inputs={'var_names': InputDescription(title='Variable names', description='Comma-separated list of variable names.', keywords=None, metadata=None, additionalParameters=AdditionalParameters(title=None, role=None, href=None, parameters=[AdditionalParameter(name='level', value=['advanced'])]), minOccurs=1, maxOccurs=None, schema_=Schema(field_ref=None, title=None, multipleOf=None, maximum=None, exclusiveMaximum=False, minimum=None, exclusiveMinimum=False, maxLength=None, minLength=0, pattern=None, maxItems=None, minItems=0, uniqueItems=False, maxProperties=No

Execute the process to receive a job ID (and other job state information):

In [5]:
job_info = client.execute_process(process_id="simulate_scene", request={"inputs": {"output_path": "test.zarr"}})
job_info

JobInfo(processID='simulate_scene', type=<JobType.process: 'process'>, jobID='job_2', status=<JobStatus.accepted: 'accepted'>, message=None, created=datetime.datetime(2026, 3, 5, 15, 23, 32, 449403, tzinfo=TzInfo(0)), started=None, finished=None, updated=None, progress=None, links=None, traceback=None)

In [6]:
job_id = job_info.jobID

Getting the job results will return an object that just points the output, but will not actually open it:

In [8]:
client.get_job_results(job_id)

JobResults(root={'return_value': InlineOrRefValue(root=InlineValue(root={'href': 'file:///C:/Users/Norman/Projects/eozilla/notebooks/test.zarr', 'rel': None, 'type': 'application/zarr', 'hreflang': None, 'title': None}))})

But if we try the open the job's result, we expect this to fail, because no openers are registered yet.

In [9]:
try:
    dataset = client.open_job_result(job_id, timeout=5)
except Exception as e:
    print(f"error: {e}")

error: No job result openers registered


Therefore we create and register a new job result opener:

In [10]:
import xarray as xr
from cuiman.api.opener import JobResultOpener, JobResultOpenContext

class MyZarrOpener(JobResultOpener):
    """Open Zarr datasets from link results."""
    
    async def accept(self, ctx: JobResultOpenContext):
        """Check if we can open the given job result.""" 
        return ctx.output_link.type == "application/zarr"
    
    async def open(self, ctx):
        """Open the given job result.""" 
        return xr.open_zarr(ctx.output_link.href, **ctx.options)


In [11]:
client.config.opener_registry.clear()
client.config.opener_registry.register(MyZarrOpener())

<function cuiman.api.opener.registry.JobResultOpenerRegistry.register.<locals>.unregister()>

Opening the job result should now work as expected:

In [12]:
dataset = client.open_job_result(job_id)
dataset

<xarray.Dataset> Size: 193MB
Dimensions:  (time: 31, lat: 360, lon: 720)
Coordinates:
  * time     (time) datetime64[ns] 248B 2025-01-01 2025-01-02 ... 2025-01-31
  * lat      (lat) float64 3kB -89.75 -89.25 -88.75 -88.25 ... 88.75 89.25 89.75
  * lon      (lon) float64 6kB -179.8 -179.2 -178.8 -178.2 ... 178.8 179.2 179.8
Data variables:
    a        (time, lat, lon) float64 64MB dask.array<chunksize=(31, 360, 720), meta=np.ndarray>
    b        (time, lat, lon) float64 64MB dask.array<chunksize=(31, 360, 720), meta=np.ndarray>
    c        (time, lat, lon) float64 64MB dask.array<chunksize=(31, 360, 720), meta=np.ndarray>